# Ingestion Pipeline


In [ ]:
import os
import sys
import json
import pandas as pd
from pathlib import Path

# Add src to path
sys.path.insert(0, 'src')

import glob
from abbreviator import Abbreviator
from create_docs import prepare_docs
from upload_dag import create_upload_dag_sh
from process_se15_file import map_se15_files_to_dataframe
from create_dags import prepare_dag_file, prepare_sql_file
from utils import check_and_remove_duplicates, add_derived_columns
from parser import parse_all_se15_files, save_to_json, DataSensitivityClassifier
from bucket import generate_bucket_input_csv, parse_final_bucket_info, merge_bucket_ids
from config import (SE15_FILES_DIR, INPUT_CSV, SE15_TABLES_JSON,OUTPUT_CONFIGS_DIR, WORD_ABBREVIATIONS_JSON,get_bucket_paths, ensure_dir_exists)

# Pandas display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.width', 1000)

# Auto-reload modules when they change (use this if you're making changes to source files)
%load_ext autoreload
%autoreload 2

---
# STEP 1: Parse SE15 Files

Parse SAP SE15 table definition files to extract:
- Table names and descriptions
- Column definitions
- Data types and constraints

Then generate:
- Abbreviated table names
- Data sensitivity classifications

In [ ]:
# Parse all SE15 files
results = parse_all_se15_files(str(SE15_FILES_DIR))

if results['processed_tables'] == 0:
    raise Exception(" No tables parsed. Check SE15 files directory.")

# Save initial results
save_to_json(results['tables'], str(SE15_TABLES_JSON))

### Generate Table Abbreviations

In [ ]:
abbr = Abbreviator(str(WORD_ABBREVIATIONS_JSON))
for table in results['tables']:
    table_name = table.get('table_name', '')
    table_desc = table.get('table_description', '')
    table['table_abbrev'] = abbr.generate_abbreviation(
        table_desc, 
        context=table_name, 
        is_table=True
    )
    for column in table.get('columns', []):
        col_desc = column.get('description', '')
        column['column_abbrev'] = abbr.generate_abbreviation(
            col_desc,
            context=None,
            is_table=False
        )
print(f"   Generated abbreviations for {results['processed_tables']} tables")

# Show sample abbreviations
print("\n Sample Table Abbreviations:")
for table in results['tables'][:3]:
    print(f"  {table['table_name']:20} → {table['table_abbrev']}")

### Classify Data Sensitivity

In [ ]:
classifier = DataSensitivityClassifier()
sensitivity_stats = {'hs': 0, 'ns': 0, 'se': 0}

for table in results['tables']:
    sensitivity = classifier.classify_table(table)
    table['sensitivity_level'] = sensitivity
    sensitivity_stats[sensitivity] += 1

print(f"  Classified {results['processed_tables']} tables:")
print(f"    - Highly Sensitive (hs): {sensitivity_stats['hs']}")
print(f"    - Non-Sensitive (ns):    {sensitivity_stats['ns']}")
print(f"    - Needs Evaluation (se): {sensitivity_stats['se']}")

# Save final version with abbreviations and sensitivity
save_to_json(results['tables'], str(SE15_TABLES_JSON))

---
# STEP 2: Load & Merge Configuration

Merge SE15 metadata with ingestion configuration.

In [ ]:
# Load SE15 metadata
print(f"\n Loading SE15 metadata from {SE15_TABLES_JSON}")
with open(SE15_TABLES_JSON, 'r', encoding='utf-8') as f:
    data = json.load(f)

# Create dataframe with SE15 data including columns
tables_data = []
for table in data:
    # Extract column mapping (column_name -> column_abbrev)
    column_mapping = {}
    for column in table.get('columns', []):
        col_name = column.get('column_name', '')
        col_abbrev = column.get('column_abbrev', '')
        if col_name and col_abbrev:
            column_mapping[col_name] = col_abbrev
    
    tables_data.append({
        'icdsTableName': table['table_name'],
        'dlTableName': table['table_abbrev'],
        'table_description': table['table_description'],
        'dataSensitivity': table['sensitivity_level'],
        'column_mapping': column_mapping,  # Add column mapping
        'columns': table.get('columns', [])  # Add full column data for SQL generation
    })

df_with_table_name = pd.DataFrame(tables_data)
print(f"   Loaded {len(df_with_table_name)} tables from SE15")

# Load ingestion CSV
print(f"\n Loading ingestion configuration from {INPUT_CSV}")
df_without_table_name = pd.read_csv(INPUT_CSV)
print(f"   Loaded {len(df_without_table_name)} records from CSV")

# Preview ingestion config
print("\n Ingestion Config Sample:")
display(df_without_table_name.head(3))


### Check for Duplicates

In [ ]:
df_without_table_name, duplicate_records, removed_count, dup_summary = check_and_remove_duplicates(
    df_without_table_name, 
    column='icdsTableName', 
    keep='first'
)

### Merge Dataframes

In [ ]:
print("\n Merging SE15 metadata with ingestion config...")
df = df_with_table_name.merge(
    df_without_table_name, 
    on='icdsTableName', 
    how='left'
)
print(f"   Merged dataframe has {len(df)} rows")

# Map SE15 files
print("\n Mapping SE15 files...")
df = map_se15_files_to_dataframe(df)

se15_count = df['se15_file_exists'].sum()
no_se15_count = len(df) - se15_count
print(f"   Records with SE15 files: {se15_count}")
print(f"    Records without SE15 files: {no_se15_count}")

# Filter to only rows with SE15 files
df = df.loc[df['se15_file_exists']].reset_index(drop=True)
print(f"\n Processing {len(df)} tables with SE15 files")

# Add derived columns
print("\n  Adding derived columns...")
df = add_derived_columns(df)
print("   Added: cluster_name, table_name, dag_name, output_dir, tags")

print("\n Final Dataframe:")
display(df.head(3))

---
# STEP 3: Generate DAGs & Documentation

Generate DAG files and documentation (these don't require bucket IDs).
SQL files will be generated after bucket creation.

In [ ]:
# Generate DAG files (independent of bucket_id)
dag_count = 0
dag_errors = []

for idx, row in df.iterrows():
    try:
        prepare_dag_file(row)
        dag_count += 1
    except Exception as e:
        error_msg = f"{row.icdsTableName}: {str(e)}"
        dag_errors.append(error_msg)
        print(f"    Error: {error_msg}")

print(f"   Generated {dag_count} DAG files")
if dag_errors:
    print(f"    {len(dag_errors)} errors occurred")

# Generate documentation (independent of bucket_id)
docs_count = 0
docs_errors = []

for idx, row in df.iterrows():
    try:
        prepare_docs(row)
        docs_count += 1
    except Exception as e:
        error_msg = f"{row.icdsTableName}: {str(e)}"
        docs_errors.append(error_msg)
        print(f"    Error: {error_msg}")

print(f"   Generated documentation for {docs_count} tables")
if docs_errors:
    print(f"    {len(docs_errors)} errors occurred")


---
# STEP 4: Generate Bucket Input CSV

Create environment-specific `bucket_input.csv` file for GCS bucket creation portal.

**File naming:**
- Dev: `input/buckets/dev-bucket_input.csv`
- Prod: `input/buckets/prod-bucket_input.csv`

Each environment maintains its own bucket files with env prefix.

In [ ]:
# Ask user for environment
print("\n Select Environment:")
print("  1. dev (Development) [DEFAULT]")
print("  2. prod (Production)")
env_choice = input("\nEnter choice (1 or 2, press Enter for dev): ").strip()

if env_choice == '2':
    env = 'prod'
    print(" Selected: Production (prod)")
else:
    env = 'dev'
    if env_choice == '1':
        print(" Selected: Development (dev)")
    else:
        print(" Defaulting to: Development (dev)")

# Generate bucket input CSV with selected environment
bucket_csv = generate_bucket_input_csv(df, env=env)

if bucket_csv:
    # Preview the bucket input file
    bucket_df = pd.read_csv(bucket_csv)
    print("\n Bucket Input Preview:")
    display(bucket_df.head(3))
    print(f"\nTotal tables: {len(bucket_df)}")
    
    print("\n" + "="*80)
    print(" BUCKET INPUT FILE GENERATED")
    print("="*80)
    print(f"\n File saved at: {bucket_csv}")
    print("\n NEXT: Complete the MANUAL STEP below before proceeding to STEP 5")
    print("="*80)
else:
    print("\n Failed to generate bucket_input.csv")
    raise Exception("Could not generate bucket input file")


In [ ]:



print("="*80)
print("STEP 5: Processing Bucket IDs")
print("="*80)

# Get environment-specific path (should match the env used in STEP 4)
bucket_paths = get_bucket_paths(env)
FINAL_BUCKET_INFO_CSV = bucket_paths['final_bucket_info_csv']

# Check if FinalBucketInfo.csv exists
if not FINAL_BUCKET_INFO_CSV.exists():
    print(f"\n ERROR: FinalBucketInfo.csv not found!")
    print(f"   Expected location: {FINAL_BUCKET_INFO_CSV}")
    print("\n" + "="*80)
    print(" CANNOT PROCEED - MANUAL STEP INCOMPLETE")
    print("="*80)
    print("\nYou must complete the manual bucket creation step:")
    print("1. Upload bucket_input.csv to the portal")
    print("2. Download the FinalBucketInfo.csv response")
    print(f"3. Save it at: {FINAL_BUCKET_INFO_CSV}")
    print("\nThen re-run this cell to continue.")
    print("="*80)
    raise FileNotFoundError(f"FinalBucketInfo.csv not found at {FINAL_BUCKET_INFO_CSV}")

print(f"\n Found FinalBucketInfo.csv")

try:
    # Parse FinalBucketInfo.csv with environment
    bucket_mapping = parse_final_bucket_info(env=env)
    
    # Preview bucket mapping
    print("\n Bucket Mapping Preview:")
    display(bucket_mapping.head(10))
    
    # Drop existing bucket_id column if present (for re-runs)
    if 'bucket_id' in df.columns:
        df = df.drop(columns=['bucket_id'])
    
    # Merge bucket IDs into main dataframe
    df = merge_bucket_ids(df, bucket_mapping)
    
    # Show sample with bucket IDs
    print("\n Dataframe with Bucket IDs:")
    display(df[['icdsTableName', 'table_name', 'bucket_id']].head(10))
    
    # Regenerate JSON metadata files with bucket IDs
    print("\n Updating JSON metadata files with bucket IDs...")
    json_count = 0
    for idx, row in df.iterrows():
        try:
            from create_docs import create_column_mapping_file
            output_dir = row.output_dir
            icds = row.get('icdsTableName') if hasattr(row, 'get') else getattr(row, 'icdsTableName', None)
            docs_dir_name = f"docs-{icds}" if icds else 'docs'
            docs_dir = os.path.join(output_dir, docs_dir_name)
            create_column_mapping_file(row, docs_dir)
            json_count += 1
        except Exception as e:
            print(f"    Error updating {row.icdsTableName}: {str(e)}")
    
    print(f"   Updated {json_count} JSON metadata files with bucket IDs")
    
    print("\n Bucket IDs successfully merged!")
    
except Exception as e:
    print(f"\n Error processing FinalBucketInfo.csv: {e}")
    print("\nPlease check:")
    print("1. File format is correct (CSV with tableName, bucket_name columns)")
    print("2. File is not corrupted")
    print("3. Table names match the ones in bucket_input.csv")
    raise


---
# STEP 6: Generate SQL Files

Generate SQL files with bucket IDs from FinalBucketInfo.csv.

**Prerequisites:**
- Bucket IDs loaded from STEP 5

In [ ]:
# Check if bucket_id column exists
if 'bucket_id' not in df.columns or df['bucket_id'].isna().all():
    print("\n ERROR: No bucket IDs found in dataframe!")
    print("   Please complete STEP 5 first to load bucket IDs.")
    raise ValueError("bucket_id column is missing or empty. Run STEP 5 first.")

# Generate SQL files with bucket IDs
sql_count = 0
sql_errors = []

for idx, row in df.iterrows():
    try:
        prepare_sql_file(row)
        sql_count += 1
    except Exception as e:
        error_msg = f"{row.icdsTableName}: {str(e)}"
        sql_errors.append(error_msg)
        print(f"    Error: {error_msg}")

print(f"   Generated {sql_count} SQL files with bucket IDs")
if sql_errors:
    print(f"   {len(sql_errors)} errors occurred")



## create upload commands


In [ ]:
DEST_DAG_BUCKET = "gs://bfdaf-dags-intldlsadev-catalog/"
dag_files = sorted(glob.glob(f"{OUTPUT_CONFIGS_DIR}/**/*.py", recursive=True))
output_path = "output/upload_dag.sh"

create_upload_dag_sh(dag_files, DEST_DAG_BUCKET, output_path)
print(f" upload commands created at : ",output_path)

---
# Final Dataframe Info

In [ ]:
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nSample Records:")
display(df.head(10))

In [ ]:
# Save the dataframe in JSON format
df.to_csv("output/final_df.csv", index=False)